# KLTN: Tác động phi tuyến của đa dạng hóa xuất khẩu đến tăng trưởng kinh tế

Dữ liệu bảng 128 quốc gia, 2000–2024 (sheet `ALL_VARIABLES`). GDPG lấy từ `GDPG_128_quoc_gia_2000_2024 (1).xlsx`: tăng trưởng GDP **bình quân đầu người** (World Bank NY.GDP.PCAP.KD.ZG), đúng định nghĩa của bài mẫu, thay cho GDPG cũ (GDP growth) trong file tổng hợp. Notebook chạy đủ 9 bước trong `target/CHECKLIST_KLTN.md`, mỗi bước ra một bảng. Chạy **Run All** từ trên xuống; toàn bộ bảng được lưu vào `data/output/KLTN_ket_qua.xlsx` ở ô cuối.

Mô hình: `GDPG ~ EDI + CAP + FDI + GOV + LAB`, với EDI là HHI hoặc THEIL. HHI/THEIL càng cao thì xuất khẩu càng **tập trung** (ít đa dạng), nên hệ số âm hàm ý đa dạng hóa hỗ trợ tăng trưởng.

Ký hiệu: `***` p<0,01; `**` p<0,05; `*` p<0,1. Sai số chuẩn trong ngoặc.

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from linearmodels.panel import PooledOLS, PanelOLS, RandomEffects

pd.set_option("display.width", 200, "display.max_columns", 30)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "input").exists())
INPUT = ROOT / "data" / "input" / "KLTN_Tong_hop_hoan_chinh.xlsx"
GDPG_INPUT = ROOT / "data" / "input" / "GDPG_128_quoc_gia_2000_2024 (1).xlsx"
OUTPUT = ROOT / "data" / "output" / "KLTN_ket_qua.xlsx"

Y = "GDPG"
EDIS = ["HHI", "THEIL"]
CONTROLS = ["CAP", "FDI", "GOV", "LAB"]
N_BOOT = 300   # số lần bootstrap cho kiểm định ngưỡng (tăng lên 1000 khi chạy bản cuối)
GRID = 200     # số điểm lưới tìm ngưỡng
TRIM = 0.05    # mỗi regime giữ tối thiểu 5% quan sát
SEED = 2024
TABLES = {}    # tên sheet -> DataFrame, ghi ra Excel ở ô cuối

In [2]:
df = pd.read_excel(INPUT, sheet_name="ALL_VARIABLES")
# Thay GDPG cũ (GDP growth) bằng tăng trưởng GDP bình quân đầu người, đúng định nghĩa bài mẫu
gdpg = pd.read_excel(GDPG_INPUT, usecols=["Code", "Year", "GDPG"])
df = df.drop(columns=Y).merge(gdpg, on=["Code", "Year"], how="left", validate="1:1").sort_values(["ID", "Year"]).reset_index(drop=True)
for e in EDIS:
    df[f"{e}2"] = df[e] ** 2
    for d in ["GFC", "COVID", "DEV"]:
        df[f"{e}x{d}"] = df[e] * df[d]

N, T = df["ID"].nunique(), df["Year"].nunique()
assert len(df) == N * T and not df[[Y, *EDIS, *CONTROLS]].isna().any().any(), "Panel phải cân bằng và không thiếu số liệu"
print(f"{N} quốc gia x {T} năm = {len(df)} quan sát;",
      f"Developed: {df.loc[df.DEV == 1, 'ID'].nunique()}, Developing: {df.loc[df.DEV == 0, 'ID'].nunique()}")
df.head()

128 quốc gia x 25 năm = 3200 quan sát; Developed: 37, Developing: 91


,ID,Country,Code,Year,Group,Region,DEV,GFC,COVID,GDP,CAP,FDI,GOV,LAB,HHI,THEIL,THEIL_PROD,THEIL_MKT,GDPG,HHI2,HHIxGFC,HHIxCOVID,HHIxDEV,THEIL2,THEILxGFC,THEILxCOVID,THEILxDEV
0,1,Albania,ALB,2000,Developing,Đang phát triển – Châu Âu/Trung Á,0,0,0,3.584570e+09,34.040139,3.989321,9.731558,-1.467925,0.244930,7.139283,2.278300,4.860983,8.149968,0.059991,0.0,0.0,0.0,50.969362,0.0,0.0,0.0
1,1,Albania,ALB,2001,Developing,Đang phát triển – Châu Âu/Trung Á,0,0,0,4.059064e+09,38.228045,5.107089,10.593907,-1.801800,0.274864,7.284668,2.387903,4.896765,9.890194,0.075550,0.0,0.0,0.0,53.066388,0.0,0.0,0.0
2,1,Albania,ALB,2002,Developing,Đang phát triển – Châu Âu/Trung Á,0,0,0,4.515003e+09,38.575336,2.990031,11.236148,-0.116632,0.288752,7.370769,2.422850,4.947918,4.942623,0.083378,0.0,0.0,0.0,54.328236,0.0,0.0,0.0
3,1,Albania,ALB,2003,Developing,Đang phát triển – Châu Âu/Trung Á,0,0,0,5.801712e+09,36.722128,3.068687,11.050514,-1.059497,0.288017,7.318967,2.378376,4.940590,5.728106,0.082954,0.0,0.0,0.0,53.567278,0.0,0.0,0.0
4,1,Albania,ALB,2004,Developing,Đang phát triển – Châu Âu/Trung Á,0,0,0,7.406646e+09,36.331947,4.607823,11.188577,-1.143219,0.279959,7.232003,2.328293,4.903711,5.707123,0.078377,0.0,0.0,0.0,52.301867,0.0,0.0,0.0


In [3]:
def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""


def fit(data, xs, kind="FE", cov="clustered"):
    """Ước lượng Pooled / FE / RE. cov: clustered (cluster theo quốc gia) hoặc unadjusted."""
    d = data.set_index(["ID", "Year"])
    kw = dict(cov_type="clustered", cluster_entity=True) if cov == "clustered" else dict(cov_type="unadjusted")
    if kind == "FE":
        return PanelOLS(d[Y], d[xs], entity_effects=True).fit(**kw)
    X = sm.add_constant(d[xs])
    return (PooledOLS if kind == "Pooled" else RandomEffects)(d[Y], X).fit(**kw)


def bang(models):
    """Gộp nhiều kết quả hồi quy thành một bảng: hệ số kèm sao, sai số chuẩn ở dòng dưới."""
    names = list(dict.fromkeys(v for r in models.values() for v in r.params.index))
    names = [v for v in names if v != "const"] + [v for v in names if v == "const"]
    index, cols = [], {m: [] for m in models}
    for v in names:
        index += [v, ""]
        for m, r in models.items():
            if v in r.params.index:
                cols[m] += [f"{r.params[v]:.4f}{stars(r.pvalues[v])}", f"({r.std_errors[v]:.4f})"]
            else:
                cols[m] += ["", ""]
    index += ["Số quan sát", "R2"]
    for m, r in models.items():
        cols[m] += [f"{int(r.nobs)}", f"{r.rsquared:.4f}"]
    return pd.DataFrame(cols, index=index)


def luu(ten, bang_kq):
    TABLES[ten] = bang_kq
    return bang_kq

## Bước 1. Thống kê mô tả

Xem trung bình, độ lệch chuẩn, nhỏ nhất, lớn nhất để mô tả đặc điểm mẫu.

In [4]:
VARS = [Y, *EDIS, *CONTROLS]
luu("1_ThongKeMoTa", df[VARS].describe().T[["count", "mean", "std", "min", "max"]]
    .rename(columns={"count": "Số quan sát", "mean": "Trung bình", "std": "Độ lệch chuẩn", "min": "Nhỏ nhất", "max": "Lớn nhất"}).round(4))

,Số quan sát,Trung bình,Độ lệch chuẩn,Nhỏ nhất,Lớn nhất
GDPG,3200.0,2.2827,4.6770,-55.2936,74.9210
HHI,3200.0,0.2916,0.1958,0.0507,0.9761
THEIL,3200.0,6.0313,1.5545,2.9862,10.3899
CAP,3200.0,24.2247,7.3568,1.0968,76.7823
FDI,3200.0,5.7979,24.5952,-296.0133,452.2209
GOV,3200.0,15.6521,5.0409,2.0484,36.1679
LAB,3200.0,1.6745,2.3575,-19.9788,25.8612


## Bước 2. Tương quan và đa cộng tuyến

Không cặp biến độc lập nào được tương quan quá cao (|r| > 0,8), VIF < 10. HHI và THEIL là hai thước đo thay thế nhau nên không đưa cùng lúc vào một mô hình; VIF tính riêng cho từng bộ biến.

In [5]:
luu("2_TuongQuan", df[VARS].corr().round(3))

,GDPG,HHI,THEIL,CAP,FDI,GOV,LAB
GDPG,1.000,-0.056,-0.042,0.198,0.060,-0.171,0.039
HHI,-0.056,1.000,0.841,0.018,-0.008,-0.145,0.303
THEIL,-0.042,0.841,1.000,0.002,-0.001,-0.291,0.294
CAP,0.198,0.018,0.002,1.000,0.021,-0.011,0.006
FDI,0.060,-0.008,-0.001,0.021,1.000,0.016,0.037
GOV,-0.171,-0.145,-0.291,-0.011,0.016,1.000,-0.171
LAB,0.039,0.303,0.294,0.006,0.037,-0.171,1.000


In [6]:
def vif(xs):
    X = sm.add_constant(df[xs])
    return pd.Series([variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])], index=xs)

luu("2b_VIF", pd.DataFrame({f"Mô hình {e}": vif([e, *CONTROLS]).rename({e: "EDI"}) for e in EDIS}).round(3))

,Mô hình HHI,Mô hình THEIL
EDI,1.113,1.171
CAP,1.001,1.001
FDI,1.003,1.002
GOV,1.041,1.103
LAB,1.124,1.106


## Bước 3. Mô hình tuyến tính cơ sở: Pooled OLS, FE, RE

Sai số chuẩn cluster theo quốc gia.

In [7]:
lin = {(e, k): fit(df, [e, *CONTROLS], k) for e in EDIS for k in ["Pooled", "FE", "RE"]}
luu("3_TuyenTinh", bang({f"{e} - {k}": r for (e, k), r in lin.items()}))

,HHI - Pooled,HHI - FE,HHI - RE,THEIL - Pooled,THEIL - FE,THEIL - RE
HHI,-2.2834***,2.5131*,-1.9981***,,,
,(0.8121),(1.3622),(0.7287),,,
CAP,0.1248***,0.1210***,0.1230***,0.1237***,0.1197***,0.1227***
,(0.0245),(0.0289),(0.0257),(0.0246),(0.0292),(0.0256)
FDI,0.0108**,0.0070,0.0090*,0.0110**,0.0069,0.0091*
,(0.0046),(0.0052),(0.0049),(0.0046),(0.0053),(0.0049)
GOV,-0.1644***,-0.3695***,-0.1945***,-0.1812***,-0.3657***,-0.2107***
,(0.0285),(0.0995),(0.0372),(0.0283),(0.1010),(0.0377)
LAB,0.0690,0.2422***,0.1561**,0.0691,0.2437***,0.1597**
,(0.0791),(0.0832),(0.0781),(0.0786),(0.0833),(0.0785)


## Bước 4. Lựa chọn mô hình panel

- **F-test** (H0: không có hiệu ứng cố định): p < 0,05 thì FE tốt hơn Pooled.
- **Breusch–Pagan LM** (H0: phương sai hiệu ứng ngẫu nhiên = 0): p < 0,05 thì RE tốt hơn Pooled.
- **Hausman** (H0: RE nhất quán): p < 0,05 thì chọn FE.

Ba kiểm định dùng ma trận hiệp phương sai thường (không robust) theo đúng giả định của kiểm định.

In [8]:
def chon_mo_hinh(e):
    xs = [e, *CONTROLS]
    po, fe, re = (fit(df, xs, k, cov="unadjusted") for k in ["Pooled", "FE", "RE"])
    # Breusch-Pagan LM cho panel cân bằng
    u = po.resids
    lm = N * T / (2 * (T - 1)) * ((u.groupby(level=0).sum() ** 2).sum() / (u ** 2).sum() - 1) ** 2
    # Hausman
    b = fe.params[xs] - re.params[xs]
    h = float(b @ np.linalg.pinv(fe.cov.loc[xs, xs] - re.cov.loc[xs, xs]) @ b)
    p_f, p_lm, p_h = fe.f_pooled.pval, stats.chi2.sf(lm, 1), stats.chi2.sf(h, len(xs))
    chon = "FE" if p_h < 0.05 and p_f < 0.05 else "RE" if p_lm < 0.05 else "Pooled OLS"
    return {"F-test": fe.f_pooled.stat, "p (F)": p_f, "BP LM": lm, "p (LM)": p_lm,
            "Hausman": h, "p (Hausman)": p_h, "Chọn": chon}

sel = pd.DataFrame({e: chon_mo_hinh(e) for e in EDIS}).T
luu("4_ChonMoHinh", sel)

,F-test,p (F),BP LM,p (LM),Hausman,p (Hausman),Chọn
HHI,3.488683,0.0,170.64372,0.0,123.015792,0.0,FE
THEIL,3.423812,0.0,174.470696,0.0,112.797607,0.0,FE


Các bước sau dùng FE (hiệu ứng cố định theo quốc gia) với sai số chuẩn cluster. Nếu bảng trên chọn mô hình khác FE thì đổi `kind` trong hàm `fit`.

## Bước 5. Mô hình phi tuyến bậc hai

`GDPG ~ EDI + EDI² + controls`. Xem đồng thời hệ số EDI (β₁) và EDI² (β₂): β₂ > 0 là dạng U, β₂ < 0 là U ngược. Điểm ngoặt = −β₁ / (2β₂), chỉ có nghĩa khi nằm trong khoảng giá trị thực của EDI.

In [9]:
quad = {e: fit(df, [e, f"{e}2", *CONTROLS]) for e in EDIS}
luu("5_BacHai", bang({f"{e} - FE bậc hai": r for e, r in quad.items()}))

,HHI - FE bậc hai,THEIL - FE bậc hai
HHI,8.2386*,
,(4.3751),
HHI2,-6.4606,
,(5.1253),
CAP,0.1202***,0.1189***
,(0.0289),(0.0294)
FDI,0.0071,0.0069
,(0.0052),(0.0052)
GOV,-0.3719***,-0.3703***
,(0.0997),(0.1012)


In [10]:
def diem_ngoat(r, e, data):
    b1, b2 = r.params[e], r.params[f"{e}2"]
    tp = -b1 / (2 * b2)
    lo, hi = data[e].min(), data[e].max()
    return {"β1 (EDI)": b1, "p (β1)": r.pvalues[e], "β2 (EDI²)": b2, "p (β2)": r.pvalues[f"{e}2"],
            "Dạng": "U" if b2 > 0 else "U ngược", "Điểm ngoặt": tp, "EDI min": lo, "EDI max": hi,
            "Trong khoảng dữ liệu": lo < tp < hi,
            "% quan sát dưới điểm ngoặt": round(100 * (data[e] < tp).mean(), 1),
            "Phi tuyến có ý nghĩa (5%)": r.pvalues[f"{e}2"] < 0.05}

luu("5b_DiemNgoat", pd.DataFrame({e: diem_ngoat(quad[e], e, df) for e in EDIS}).T)

,β1 (EDI),p (β1),β2 (EDI²),p (β2),Dạng,Điểm ngoặt,EDI min,EDI max,Trong khoảng dữ liệu,% quan sát dưới điểm ngoặt,Phi tuyến có ý nghĩa (5%)
HHI,8.23858,0.059785,-6.460622,0.207576,U ngược,0.6376,0.050725,0.97614,True,92.4,False
THEIL,3.755382,0.001859,-0.241421,0.00359,U ngược,7.777657,2.986203,10.389943,True,86.1,True


## Bước 6. Panel threshold regression (Hansen, 1999)

Biến ngưỡng là chính EDI; hệ số của EDI thay đổi theo regime, các biến kiểm soát giữ chung:

`GDPG = μᵢ + β₁·EDI·1(EDI ≤ γ) + β₂·EDI·1(EDI > γ) + θ·controls + ε`

- Ngưỡng γ tìm bằng cách tối thiểu tổng bình phương phần dư trên lưới phân vị 5%–95% của EDI.
- Kiểm định có ngưỡng hay không (H0: không có ngưỡng) bằng thống kê F với p-value bootstrap.

In [11]:
def _within(a, n, t):
    a = a.reshape(n, t, -1)
    return (a - a.mean(1, keepdims=True)).reshape(n * t, -1)


def _Q(p, g=None):
    # Cơ sở trực chuẩn của biến giải thích sau khi khử hiệu ứng cố định; g=None là mô hình không ngưỡng
    q = p["q"]
    R = q[:, None] if g is None else np.column_stack([q * (q <= g), q * (q > g)])
    return np.linalg.qr(np.hstack([_within(R, p["n"], p["t"]), p["C"]]))[0]


def _ssr(Q, Yv):
    return (Yv ** 2).sum(0) - ((Q.T @ Yv) ** 2).sum(0)


def threshold(data, e):
    d = data.sort_values(["ID", "Year"])
    n = d["ID"].nunique(); t = len(d) // n
    p = {"n": n, "t": t, "q": d[e].to_numpy(float), "C": _within(d[CONTROLS].to_numpy(float), n, t)}
    y = _within(d[[Y]].to_numpy(float), n, t)[:, 0]
    dof = n * (t - 1)
    grid = np.quantile(p["q"], np.linspace(TRIM, 1 - TRIM, GRID))

    Q0, Q1 = _Q(p), [_Q(p, g) for g in grid]
    s1_all = np.array([_ssr(Q, y) for Q in Q1])
    i = int(s1_all.argmin()); s1 = s1_all[i]; g = grid[i]
    F = (_ssr(Q0, y) - s1) / (s1 / dof)

    # Bootstrap: lấy mẫu lại nguyên chuỗi phần dư của từng quốc gia
    resid = (y - Q1[i] @ (Q1[i].T @ y)).reshape(n, t)
    Yb = resid[np.random.default_rng(SEED).integers(0, n, (N_BOOT, n))].reshape(N_BOOT, -1).T
    s1b = np.array([_ssr(Q, Yb) for Q in Q1]).min(0)
    pval = float(((_ssr(Q0, Yb) - s1b) / (s1b / dof) >= F).mean())

    dd = d.copy()
    dd[f"{e}_duoi_nguong"], dd[f"{e}_tren_nguong"] = dd[e] * (dd[e] <= g), dd[e] * (dd[e] > g)
    tom_tat = {"Ngưỡng (γ)": g, "% quan sát ≤ γ": round(100 * (p["q"] <= g).mean(), 1),
               "F": F, "p bootstrap": pval, "Có ngưỡng (5%)": pval < 0.05}
    return tom_tat, fit(dd, [f"{e}_duoi_nguong", f"{e}_tren_nguong", *CONTROLS])

In [12]:
thr = {e: threshold(df, e) for e in EDIS}
luu("6_Nguong", pd.DataFrame({e: thr[e][0] for e in EDIS}).T)

,Ngưỡng (γ),% quan sát ≤ γ,F,p bootstrap,Có ngưỡng (5%)
HHI,0.211201,43.0,6.898912,0.593333,False
THEIL,7.691825,85.0,6.191611,0.823333,False


In [13]:
luu("6b_HeSoRegime", bang({f"{e} - threshold": thr[e][1] for e in EDIS}))

,HHI - threshold,THEIL - threshold
HHI_duoi_nguong,-3.0835,
,(4.6380),
HHI_tren_nguong,1.5858,
,(1.6166),
CAP,0.1218***,0.1209***
,(0.0289),(0.0294)
FDI,0.0070,0.0070
,(0.0051),(0.0052)
GOV,-0.3727***,-0.3678***
,(0.1004),(0.1018)


`duoi_nguong` là hệ số của EDI khi EDI ≤ γ (đa dạng hóa cao), `tren_nguong` là khi EDI > γ. Chỉ diễn giải hai hệ số này khi kiểm định ngưỡng có ý nghĩa.

## Bước 7. So sánh Developed và Developing

Chạy riêng từng nhóm (tuyến tính và threshold), rồi dùng biến tương tác `EDI×DEV` trên toàn mẫu để kiểm định khác biệt hệ số có ý nghĩa thống kê hay không (DEV = 1 là nước phát triển; bản thân DEV bị hiệu ứng cố định hấp thụ).

In [14]:
GROUPS = {"Developed": df[df.DEV == 1], "Developing": df[df.DEV == 0]}
grp_lin = {(e, g): fit(d, [e, *CONTROLS]) for e in EDIS for g, d in GROUPS.items()}
luu("7_NhomNuoc_HoiQuy", bang({f"{e} - {g}": r for (e, g), r in grp_lin.items()}))

,HHI - Developed,HHI - Developing,THEIL - Developed,THEIL - Developing
HHI,10.5827***,1.8469,,
,(3.7166),(1.4532),,
CAP,0.2595***,0.0521,0.2309***,0.0511
,(0.0734),(0.0423),(0.0650),(0.0424)
FDI,0.0033,0.1640***,0.0025,0.1653***
,(0.0027),(0.0593),(0.0029),(0.0599)
GOV,-0.7679***,-0.2902***,-0.7992***,-0.2872***
,(0.0957),(0.1044),(0.0933),(0.1059)
LAB,0.1762,0.2315***,0.1754,0.2330***
,(0.1162),(0.0887),(0.1166),(0.0887)


In [15]:
grp_thr = {(e, g): threshold(d, e) for e in EDIS for g, d in GROUPS.items()}
luu("7b_NhomNuoc_Nguong", pd.DataFrame({f"{e} - {g}": r[0] for (e, g), r in grp_thr.items()}).T)

,Ngưỡng (γ),% quan sát ≤ γ,F,p bootstrap,Có ngưỡng (5%)
HHI - Developed,0.090699,23.6,8.462493,0.383333,False
HHI - Developing,0.211503,28.5,7.500819,0.38,False
THEIL - Developed,5.640342,82.7,8.518349,0.526667,False
THEIL - Developing,7.050735,61.5,13.294524,0.146667,False


In [16]:
luu("7c_NhomNuoc_Regime", bang({f"{e} - {g}": r[1] for (e, g), r in grp_thr.items()}))

,HHI - Developed,HHI - Developing,THEIL - Developed,THEIL - Developing
HHI_duoi_nguong,26.3654***,-5.0545,,
,(6.6493),(5.1136),,
HHI_tren_nguong,12.3984***,0.7990,,
,(3.8589),(1.6919),,
CAP,0.2702***,0.0524,0.2273***,0.0500
,(0.0759),(0.0421),(0.0640),(0.0421)
FDI,0.0037,0.1618***,0.0025,0.1723***
,(0.0028),(0.0576),(0.0028),(0.0609)
GOV,-0.7893***,-0.2946***,-0.8074***,-0.2684***
,(0.0990),(0.1057),(0.0958),(0.1007)


In [17]:
inter = {e: fit(df, [e, f"{e}xDEV", *CONTROLS]) for e in EDIS}
luu("7d_TuongTacDEV", bang({f"{e} - FE tương tác DEV": r for e, r in inter.items()}))

,HHI - FE tương tác DEV,THEIL - FE tương tác DEV
HHI,1.7057,
,(1.4170),
HHIxDEV,8.1739*,
,(4.5763),
CAP,0.1217***,0.1182***
,(0.0293),(0.0292)
FDI,0.0080,0.0075
,(0.0052),(0.0054)
GOV,-0.3702***,-0.3721***
,(0.0996),(0.1012)


Hệ số `EDI` là tác động ở nhóm Developing; `EDIxDEV` là chênh lệch của nhóm Developed so với Developing. `EDIxDEV` có ý nghĩa thì hai nhóm khác nhau có ý nghĩa thống kê.

## Bước 8. Khủng hoảng GFC (2008–2009) và COVID (2020–2021)

`GDPG ~ EDI + EDI×GFC + EDI×COVID + GFC + COVID + controls`. Hệ số tương tác cho biết tác động của EDI thay đổi bao nhiêu trong năm khủng hoảng.

In [18]:
crisis = {e: fit(df, [e, f"{e}xGFC", f"{e}xCOVID", "GFC", "COVID", *CONTROLS]) for e in EDIS}
luu("8_KhungHoang", bang({f"{e} - FE khủng hoảng": r for e, r in crisis.items()}))

,HHI - FE khủng hoảng,THEIL - FE khủng hoảng
HHI,1.8334,
,(1.3181),
HHIxGFC,3.5439**,
,(1.3909),
HHIxCOVID,-0.9777,
,(1.5616),
GFC,-3.6223***,-5.8483***
,(0.4730),(1.0295)
COVID,-1.5703***,0.2287
,(0.4572),(0.7957)


## Bước 9. Kiểm định độ vững

- Thay HHI ↔ THEIL: đã chạy song song ở mọi bước, so sánh hai cột.
- Sai số chuẩn: mọi mô hình FE ở trên đã dùng sai số chuẩn cluster theo quốc gia.
- Outlier: đếm quan sát lệch quá 3 độ lệch chuẩn, rồi chạy lại sau khi winsorize 1%–99% toàn bộ biến liên tục.

In [19]:
z = (df[VARS] - df[VARS].mean()) / df[VARS].std()
luu("9_Outlier", pd.DataFrame({"Số quan sát |z| > 3": (z.abs() > 3).sum(), "Tỷ lệ (%)": (100 * (z.abs() > 3).mean()).round(2)}))

,Số quan sát |z| > 3,Tỷ lệ (%)
GDPG,40,1.25
HHI,42,1.31
THEIL,0,0.00
CAP,34,1.06
FDI,29,0.91
GOV,9,0.28
LAB,60,1.88


In [20]:
dfw = df.copy()
dfw[VARS] = dfw[VARS].clip(dfw[VARS].quantile(0.01), dfw[VARS].quantile(0.99), axis=1)
for e in EDIS:
    dfw[f"{e}2"] = dfw[e] ** 2

rob = {}
for e in EDIS:
    for ten, xs in {"tuyến tính": [e, *CONTROLS], "bậc hai": [e, f"{e}2", *CONTROLS]}.items():
        rob[f"{e} {ten} - cluster"] = fit(df, xs)
        rob[f"{e} {ten} - winsorize"] = fit(dfw, xs)
luu("9b_DoVung", bang(rob))

,HHI tuyến tính - cluster,HHI tuyến tính - winsorize,HHI bậc hai - cluster,HHI bậc hai - winsorize,THEIL tuyến tính - cluster,THEIL tuyến tính - winsorize,THEIL bậc hai - cluster,THEIL bậc hai - winsorize
HHI,2.5131*,2.3388*,8.2386*,6.7840*,,,,
,(1.3622),(1.2224),(4.3751),(3.9661),,,,
CAP,0.1210***,0.1141***,0.1202***,0.1138***,0.1197***,0.1125***,0.1189***,0.1130***
,(0.0289),(0.0222),(0.0289),(0.0222),(0.0292),(0.0225),(0.0294),(0.0227)
FDI,0.0070,0.0493**,0.0071,0.0491**,0.0069,0.0496**,0.0069,0.0483**
,(0.0052),(0.0212),(0.0052),(0.0211),(0.0053),(0.0214),(0.0052),(0.0212)
GOV,-0.3695***,-0.3219***,-0.3719***,-0.3238***,-0.3657***,-0.3186***,-0.3703***,-0.3225***
,(0.0995),(0.0616),(0.0997),(0.0618),(0.1010),(0.0638),(0.1012),(0.0640)
LAB,0.2422***,0.2381***,0.2418***,0.2376***,0.2437***,0.2385***,0.2432***,0.2386***
,(0.0832),(0.0675),(0.0836),(0.0676),(0.0833),(0.0676),(0.0834),(0.0674)


## Tóm tắt để trả lời 5 câu hỏi (phần B của checklist)

In [21]:
def tom_tat(e):
    l, q, t, i = lin[e, "FE"], quad[e], thr[e][0], inter[e]
    return {
        "1. Hệ số EDI (FE tuyến tính)": f"{l.params[e]:.4f}{stars(l.pvalues[e])} (p={l.pvalues[e]:.3f})",
        "2. Chiều tác động": ("Tập trung cao làm giảm tăng trưởng" if l.params[e] < 0 else "Tập trung cao đi kèm tăng trưởng cao hơn")
                             + ("" if l.pvalues[e] < 0.05 else " (không có ý nghĩa 5%)"),
        "3. Phi tuyến (EDI²)": f"{q.params[e + '2']:.4f}{stars(q.pvalues[e + '2'])}, dạng {'U' if q.params[e + '2'] > 0 else 'U ngược'}, "
                               f"điểm ngoặt {-q.params[e] / (2 * q.params[e + '2']):.4f}",
        "4. Ngưỡng": f"γ={t['Ngưỡng (γ)']:.4f}, p bootstrap={t['p bootstrap']:.3f}",
        "5. Khác biệt Developed (EDIxDEV)": f"{i.params[e + 'xDEV']:.4f}{stars(i.pvalues[e + 'xDEV'])} (p={i.pvalues[e + 'xDEV']:.3f})",
    }

pd.set_option("display.max_colwidth", None)
luu("0_TomTat", pd.DataFrame({e: tom_tat(e) for e in EDIS}))

,HHI,THEIL
1. Hệ số EDI (FE tuyến tính),2.5131* (p=0.065),0.4650* (p=0.063)
2. Chiều tác động,Tập trung cao đi kèm tăng trưởng cao hơn (không có ý nghĩa 5%),Tập trung cao đi kèm tăng trưởng cao hơn (không có ý nghĩa 5%)
3. Phi tuyến (EDI²),"-6.4606, dạng U ngược, điểm ngoặt 0.6376","-0.2414***, dạng U ngược, điểm ngoặt 7.7777"
4. Ngưỡng,"γ=0.2112, p bootstrap=0.593","γ=7.6918, p bootstrap=0.823"
5. Khác biệt Developed (EDIxDEV),8.1739* (p=0.074),1.8832** (p=0.014)


In [22]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT) as w:
    for ten in sorted(TABLES):
        TABLES[ten].to_excel(w, sheet_name=ten[:31])
print(f"Đã lưu {len(TABLES)} bảng vào {OUTPUT}")

Đã lưu 17 bảng vào c:\1_workspace\khanh\nghiencuukhoahoc\data\output\KLTN_ket_qua.xlsx


## Nhận xét: dữ liệu và kết quả so với bài nghiên cứu mẫu

Bài mẫu: Trinh & Thuy (2021), *Export diversification and economic growth: A threshold regression approach for emerging markets and developing countries*, EJEM 13(2) (`target/document.pdf`).
Các số dưới đây lấy từ lần chạy ngày 18/11/2026 với GDPG bình quân đầu người và `N_BOOT = 300`. Chạy lại với tham số khác thì số sẽ thay đổi chút ít.

### 1. Dữ liệu: giống và khác bài mẫu

| Nội dung | Bài mẫu | Dữ liệu KLTN | Nhận xét |
|---|---|---|---|
| Mẫu | 44 nước mới nổi và đang phát triển, 1995–2015 | 128 nước (37 phát triển, 91 đang phát triển), 2000–2024, panel cân bằng 3.200 quan sát | Mẫu rộng hơn nhiều, có cả nước phát triển. Nhóm Developing ở bước 7 mới là nhóm tương đương bài mẫu. |
| GDPG | Tăng trưởng GDP bình quân đầu người: TB 3,05; SD 3,52; [−14,4; 23,1] | Đã đổi sang **GDP bình quân đầu người** (WB NY.GDP.PCAP.KD.ZG), đúng định nghĩa bài mẫu: TB 2,28; SD 4,68; [−55,3; 74,9] | Định nghĩa đã khớp. Biên độ rộng hơn do các cú sốc như Macao 2020/2023, Iraq 2003/2004, CH Trung Phi 2013. GDPG có 40 quan sát (1,25%) lệch quá 3 SD. |
| HHI | Chỉ số tập trung **thị trường** (theo địa lý, phân tán theo đối tác nhập khẩu), TB 0,124, [0; 0,71] | Chỉ số tập trung **sản phẩm** của UNCTAD (Concentration Index), TB 0,292, [0,05; 0,98] | **Khác khái niệm.** HHI của bài mẫu đo đa dạng hóa thị trường, HHI ở đây đo đa dạng hóa sản phẩm, nên ngưỡng không so trực tiếp được. |
| THEIL | Overall Theil của IMF (theo sản phẩm, gồm biên độ intensive và extensive), TB 2,94, [1,60; 5,59] | Overall Theil của UNCTAD = THEIL_PROD + THEIL_MKT, TB 6,03, [2,99; 10,39] | **Khác nguồn và thang đo.** Giá trị ở đây gần gấp đôi, nên ngưỡng 2,069 của bài mẫu không áp dụng được. THEIL_PROD (TB 2,38, [0,77; 5,42]) gần với thang đo của bài mẫu hơn. |
| Biến kiểm soát | CAP, FDI, GOV, LAB, cùng dummy AFC 1997 và GFC 2008 trong mọi mô hình | CAP, FDI, GOV, LAB. GFC và COVID chỉ đưa vào ở bước 8. Không có AFC vì mẫu bắt đầu từ năm 2000. | FDI có ngoại lai rất lớn ([−296; 452]% GDP, do các trung tâm tài chính), so với [−16; 51] ở bài mẫu. |
| Phương pháp ngưỡng | Kiểm định Bai–Perron | Hansen (1999), p-value bootstrap theo cụm quốc gia | Cách của Hansen thận trọng hơn: thống kê F gần bằng bài mẫu nhưng p-value cao hơn. |

### 2. Kết quả so với bài mẫu

| Câu hỏi | Bài mẫu | KLTN (GDPG bình quân đầu người) | Khớp? |
|---|---|---|---|
| Chọn mô hình | FE (Hausman 33,5 và 15,1, p < 0,05) | FE (Hausman 123,0 và 112,8; F-test và LM đều có p ≈ 0) | ✅ |
| Có ngưỡng không | Có: HHI F = 6,47\*\*\*, THE F = 4,78\*\*; γ = 0,060 và 2,069 | **Không**: HHI F = 6,90, p = 0,59; THEIL F = 6,19, p = 0,82; γ = 0,211 và 7,692 | ❌ |
| Hệ số EDI theo regime | Âm ở cả hai regime; có ý nghĩa ở regime 2 (HHI −5,27\*\*, THE −1,59\*\*\*) | Toàn mẫu: HHI không có ý nghĩa; THEIL **dương** ở cả hai regime (0,76\*\* và 0,62\*\*). Developing: HHI −5,05 và 0,80; THEIL −0,35 và −0,10, đều không có ý nghĩa | ❌ toàn mẫu. ≈ nhóm Developing (cùng dấu âm nhưng không có ý nghĩa) |
| Bậc hai (FE) | HHI −5,01\*, HHI² −1,52\*\*; THE −6,33\*\*\*, THE² −0,69\*\* | HHI 8,24\*, HHI² −6,46 (không có ý nghĩa); THEIL 3,76\*\*\*, THEIL² −0,24\*\*\*, điểm ngoặt 7,78, có 86% quan sát nằm dưới điểm ngoặt | ❌ Ngược dấu β₁. Ở đây THEIL có dạng U ngược theo **mức tập trung**: với đa số quan sát, tập trung cao hơn đi kèm tăng trưởng cao hơn. |
| Tuyến tính | Không báo cáo | Pooled và RE: HHI và THEIL âm \*\*\* (cùng chiều bài mẫu). FE: dương, chỉ có ý nghĩa ở mức 10% | Dấu đổi khi kiểm soát hiệu ứng quốc gia: tương quan âm chủ yếu đến từ khác biệt **giữa** các nước. |
| CAP | Dương \*\*\* (0,11–0,22) | Dương \*\*\* (≈ 0,12) | ✅ |
| GOV | Âm \*\*\* (−0,34 đến −0,47) | Âm \*\*\* (≈ −0,37) | ✅ |
| LAB | Dương có ý nghĩa trong mô hình ngưỡng | Dương \*\*\* (≈ 0,24) | ✅ |
| FDI | Dương có ý nghĩa (0,03–0,21) | Toàn mẫu không có ý nghĩa; Developing 0,16\*\*\*; sau winsorize 0,05\*\* | ✅ khi bỏ ảnh hưởng ngoại lai, hoặc chỉ xét nhóm Developing (độ lớn gần bài mẫu) |
| Khủng hoảng GFC | C08 −2,29\*\*\* | GFC −3,6\*\*\* (HHI) và −5,8\*\*\* (THEIL); EDI×GFC dương có ý nghĩa | ✅ GFC kéo giảm tăng trưởng. Bài mẫu không có COVID; ở đây THEIL×COVID âm \*\*. |
| R² | 0,30–0,37 | ≈ 0,07 (within R²) | Không so trực tiếp được: bài mẫu có dummy và hằng số. Within R² luôn thấp hơn. |

### 3. Ảnh hưởng của việc đổi GDPG (GDP growth sang GDP bình quân đầu người)

- Hai chuỗi tương quan 0,957, nên kết luận chính **gần như không đổi**: vẫn chọn FE, vẫn không có ngưỡng, THEIL² vẫn âm \*\*\*, điểm ngoặt dịch rất ít (THEIL 7,85 → 7,78; HHI 0,644 → 0,638).
- Mức ý nghĩa giảm nhẹ: FE tuyến tính HHI từ \*\* (p = 0,047) xuống \* (p = 0,065), THEIL từ \*\* (p = 0,034) xuống \* (p = 0,063). HHI×DEV từ không có ý nghĩa (p = 0,16) lên \* (p = 0,074).

### 4. Kết luận và việc cần làm

1. Dữ liệu KLTN **không tái lập được** kết quả trung tâm của bài mẫu (có ngưỡng; đa dạng hóa có lợi cho tăng trưởng ở regime trên ngưỡng). Lý do chính nằm ở **dữ liệu**, không phải ở code: (i) HHI đo đa dạng hóa sản phẩm, còn bài mẫu đo đa dạng hóa thị trường; (ii) THEIL khác nguồn và thang đo; (iii) mẫu có cả nước phát triển, mà ở nhóm này hệ số EDI dương mạnh (HHI 10,58\*\*\*, THEIL 1,98\*\*\*) và kéo kết quả toàn mẫu sang dương; (iv) giai đoạn khác (2000–2024 so với 1995–2015).
2. Nhóm **Developing** (tương đương bài mẫu) cho hệ số EDI âm trong mô hình ngưỡng, cùng chiều bài mẫu, nhưng không có ý nghĩa thống kê, và kiểm định ngưỡng không bác bỏ H0.
3. Các biến kiểm soát (CAP, GOV, LAB, FDI khi bỏ ngoại lai) và tác động của GFC **khớp với bài mẫu**. Đây là điểm có thể viết là “nhất quán với Trinh & Thuy (2021)”.
4. Nên làm thêm trước khi viết chương kết quả:
   - Nếu muốn so sánh sát bài mẫu: dùng THEIL_PROD thay THEIL, và tìm chỉ số HHI theo thị trường (WITS: *HH Market concentration index*).
   - Chạy riêng nhóm Developing, giai đoạn 2000–2015, để có phép so sánh “cùng điều kiện”.
   - Tăng `N_BOOT` lên 1000 cho bản cuối. Đối chiếu lại phân loại DEV của 43 nước bổ sung (đang là giả định).
